In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# -------------------------------
# CSS for card-style panels
# -------------------------------
display(HTML("""
<style>
.panel {
    border: 1px solid #ccc;
    border-radius: 8px;
    padding: 8px;
    margin-bottom: 12px;
    background-color: #fafafa;
    box-shadow: 1px 1px 5px rgba(0,0,0,0.1);
}
.panel h4 {
    margin-top: 0px;
    margin-bottom: 6px;
}
@media (max-width: 700px) {
    .card-grid {
        display: flex;
        flex-direction: column;
        align-items: center;
    }
    .card {
        width: 95% !important;
        margin-bottom: 12px;
    }
    .radius-slider {
        width: 95% !important;
        height: 40px !important;
        margin: 10px 0 0 0 !important;
    }
}
@media (min-width: 701px) {
    .card-grid {
        display: grid;
        grid-template-columns: 1fr 1fr;
        grid-template-rows: 1fr 1fr;
        grid-gap: 12px;
    }
    .card {
        width: 100% !important;
        height: 100% !important;
    }
    .radius-slider {
        height: 100% !important;
    }
}
</style>
"""))

# -------------------------------
# Helpers
# -------------------------------
def get_uploaded_file_content(upload_widget):
    if not upload_widget.value:
        return None
    val = upload_widget.value
    if isinstance(val, dict):
        return list(val.values())[0]["content"]
    if isinstance(val, tuple):
        return val[0]["content"]
    return None

def normalize_image(img):
    img = img - np.min(img)
    if np.max(img) > 0:
        img = img / np.max(img)
    return (img * 255).astype(np.uint8)

def resize_if_needed(pil_img, max_size=512):
    if pil_img.width > max_size or pil_img.height > max_size:
        pil_img = pil_img.copy()
        pil_img.thumbnail((max_size, max_size))
    return pil_img

def to_grayscale_array(pil_img):
    if pil_img.mode != "L":
        pil_img = pil_img.convert("L")
    return np.array(pil_img, dtype=np.float32)

def circular_mask(shape, radius, filter_type):
    rows, cols = shape
    crow, ccol = rows // 2, cols // 2
    Y, X = np.ogrid[:rows, :cols]
    dist = np.sqrt((X - ccol)**2 + (Y - crow)**2)
    if filter_type == 'Low-pass':
        mask = dist <= radius
    elif filter_type == 'High-pass':
        mask = dist >= radius
    else:
        mask = np.ones_like(dist, dtype=bool)
    return mask.astype(float)

def compute_radius_from_click(event, arr_shape):
    if event.xdata is None or event.ydata is None:
        return None
    rows, cols = arr_shape
    crow, ccol = rows // 2, cols // 2
    x, y = event.xdata, event.ydata
    r = np.sqrt((x - ccol)**2 + (y - crow)**2)
    return int(r)

# -------------------------------
# Widgets
# -------------------------------
title_html = widgets.HTML("<h2>Interactive Fourier Filtering Explorer</h2>")
subtitle_html = widgets.HTML("<p>Upload an image, interact with FFT mask, and explore low/high-pass filters.</p>")

upload = widgets.FileUpload(accept="image/*", multiple=False, description="📁 Upload Image")
filter_type = widgets.Dropdown(options=['None','Low-pass','High-pass'], value='Low-pass', description='Filter:')
radius_slider = widgets.IntSlider(
    value=40,
    min=1,
    max=400,
    step=1,
    description='Radius:',
    orientation='vertical',
    continuous_update=False,
    layout=widgets.Layout(
        height='95%',
        margin='0px 5px 0px 5px'  # optional padding
    )
)
cmap_selector = widgets.Dropdown(options=['magma','inferno','viridis','gray'], value='magma', description='FFT colormap:')
out_status = widgets.Output()
out_original = widgets.Output()
out_fft = widgets.Output()
out_lowpass = widgets.Output()
out_highpass = widgets.Output()

# -------------------------------
# Processing function
# -------------------------------
def process_and_display(change=None):
    out_status.clear_output(wait=True)
    out_original.clear_output(wait=True)
    out_fft.clear_output(wait=True)
    out_lowpass.clear_output(wait=True)
    out_highpass.clear_output(wait=True)

    content = get_uploaded_file_content(upload)
    if content is None:
        with out_status:
            print("Waiting for image upload...")
        return

    try:
        with out_status:
            print("Processing image...")

        pil_img = Image.open(io.BytesIO(content))
        pil_img = resize_if_needed(pil_img)
        gray_arr = to_grayscale_array(pil_img)

        # FFT
        F = np.fft.fft2(gray_arr)
        Fshift = np.fft.fftshift(F)
        magnitude = np.abs(Fshift)
        display_magnitude = np.log1p(magnitude)

        # Masks
        rad = radius_slider.value
        mask_low = circular_mask(gray_arr.shape, rad, 'Low-pass')
        mask_high = circular_mask(gray_arr.shape, rad, 'High-pass')

        F_low = Fshift * mask_low
        F_high = Fshift * mask_high

        F_ishift_low = np.fft.ifftshift(F_low)
        F_ishift_high = np.fft.ifftshift(F_high)

        img_low = np.abs(np.fft.ifft2(F_ishift_low))
        img_high = np.abs(np.fft.ifft2(F_ishift_high))

        img_low_norm = normalize_image(img_low)
        img_high_norm = normalize_image(img_high)

        # ORIGINAL
        with out_original:
            plt.figure(figsize=(4,4))
            plt.imshow(pil_img if pil_img.mode in ("RGB","RGBA") else gray_arr, cmap='gray')
            plt.title("Original Image")
            plt.axis('off')
            plt.show()

        # FFT + Mask overlay
        with out_fft:
            fig, ax = plt.subplots(figsize=(4,4))
            ax.imshow(display_magnitude, cmap=cmap_selector.value)
            overlay = np.zeros((*mask_low.shape,4))
            overlay[mask_low==1] = [1,1,0,0.35]  # always show low-pass mask overlay
            ax.imshow(overlay)
            ax.set_title("FFT Magnitude (click to adjust radius)")
            ax.axis('off')

            def onclick(event):
                r = compute_radius_from_click(event, gray_arr.shape)
                if r is not None:
                    radius_slider.value = r
            fig.canvas.mpl_connect("button_press_event", onclick)
            plt.show()

        # LOW-PASS
        with out_lowpass:
            plt.figure(figsize=(4,4))
            plt.imshow(img_low_norm, cmap='gray')
            plt.title("Low-pass Filtered")
            plt.axis('off')
            plt.show()

        # HIGH-PASS
        with out_highpass:
            plt.figure(figsize=(4,4))
            plt.imshow(img_high_norm, cmap='gray')
            plt.title("High-pass Filtered")
            plt.axis('off')
            plt.show()

    except Exception as e:
        with out_status:
            print("Error:", e)

# -------------------------------
# Observers
# -------------------------------
upload.observe(process_and_display, names='value')
filter_type.observe(process_and_display, names='value')
radius_slider.observe(process_and_display, names='value')
cmap_selector.observe(process_and_display, names='value')

# -------------------------------
# Layout - 2x2 + vertical radius slider
# -------------------------------
# Left column: original + FFT
left_column = widgets.VBox([out_original, out_fft], layout=widgets.Layout(width='50%'))

# Right column: low-pass + high-pass
right_column = widgets.VBox([out_lowpass, out_highpass], layout=widgets.Layout(width='50%'))

# Main 2x2 grid
grid_panel = widgets.HBox([
    left_column,
    right_column,
    widgets.Box([radius_slider], layout=widgets.Layout(width='50px'))  # vertical slider
    ], 
    layout=widgets.Layout(
        width='100%',
        align_items='stretch',
        overflow='visible'  # avoids unwanted vertical scrollbar
    )
)

# Controls at top
control_panel = widgets.VBox([
    widgets.HTML("<h3>Controls</h3>"),
    upload,
    filter_type,
    cmap_selector,
    out_status
], layout=widgets.Layout(width='100%', align_items='stretch'))

# Display everything
display(widgets.VBox([
    title_html,
    subtitle_html,
    control_panel,
    widgets.HTML("<hr>"),
    grid_panel
]))
